In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False,  json.loads(sys.argv[1])['scenario'])

# START

In [2]:
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import connectionscanmodel, stepmodel
from quetzal.io import excel
import numpy as np
import pandas as pd
import geopandas as gpd

import shapely

import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

Index(['description', 'unit', 'category', 'parameter', 'base',
       'base_timetables'],
      dtype='object')


In [5]:
#if var['run_mode']['mode'] == 'timetables':
#    print('timetables scenario')
#    end_of_notebook

In [6]:
NTLEG_PENALTY = var[('pathfinders_general', 'ntleg_penalty')]
TRANSFER_DURATION = var[('pathfinders_general', 'transfer_duration')]
WALK_SPEED = var[('pathfinders_general', 'walk_speed')]
TRANSFER_PENALTY = var[('pathfinders_general', 'transfer_penalty')]

WALK_TIME_FACTOR = var[('mode_penalty', 'walk_time_factor')]
BIKE_TIME_FACTOR = var[('mode_penalty', 'bike_time_factor')]
CAR_TIME_FACTOR = var[('mode_penalty', 'car_time_factor')]
BOARDING_TIMES = var['boarding_time'].to_dict()

FLAT_FARES = var['fare'].to_dict()
KM_FARES = var['fare_km'].to_dict()

CAR_KM_FARES = var['fare_car'].to_dict()
KM_FARES['car'] = CAR_KM_FARES['fuel_cost'] + CAR_KM_FARES['other_costs']

BIKE_SPEED = var['bike_speed'].to_dict()
BIKE_FACTORS = {int(x.replace('k', '')): (1+y) for x, y in var['bike_factor'].to_dict().items()}
BIKE_FACTORS[0] = 0

VALUE_OF_TIME_SEC = var[('value_of_time', 'value_of_time_h')]/3600

INTRA_AREA_TO_DISTANCE_FACTOR = var[('step_distribution', 'intra_area_to_distance_factor')]

# READ

In [7]:
sm = stepmodel.read_zippedpickles(mpath + 'zones')

zones: 100%|██████████| 13/13 [00:00<00:00, 16.65it/s]    


# NODE CLUSTERING

In [8]:
sm.preparation_clusterize_nodes(
     distance_threshold=200
)

In [9]:
sm._add_type_prefixes(
        prefixes={'nodes': 'node_'}
    )

can not add prefixes on table:  footpaths


In [10]:
sm_bike = sm.copy()
sm_bike.road_links, sm_bike.road_nodes = sm_bike.road_links.loc[sm_bike.road_links['bike']], sm_bike.road_nodes.loc[sm_bike.road_nodes['bike']]

sm_car = sm.copy()
sm_car.road_links, sm_car.road_nodes = sm_car.road_links.loc[sm_car.road_links['bike']], sm_car.road_nodes.loc[sm_car.road_nodes['bike']]

# Walkmodels

In [11]:
sm.road_links['walk_speed'] = WALK_SPEED # km / h
sm.road_links['walk_time'] = sm.road_links.length / (sm.road_links['walk_speed'] / 3.6) * WALK_TIME_FACTOR

In [12]:
sm.preparation_ntlegs(
    zone_to_road=True,
    zone_to_transit=False,
    short_leg_speed=WALK_SPEED/WALK_TIME_FACTOR,
    long_leg_speed=WALK_SPEED/WALK_TIME_FACTOR
)

In [13]:
sm._add_type_prefixes(
    prefixes={
        'zone_to_road': 'ztr_',
        'road_to_transit': 'rtt_',
    }
)

# PATHFINDER - PT ONLY

In [14]:
# Access times
sm.links['time'] += sm.links['route_type'].map(BOARDING_TIMES).fillna(0)

In [15]:
# add fare in time for pathfinder to find most relevant paths
sm.links['km_fare'] = sm.links.length * sm.links.route_type.apply(lambda x: KM_FARES[x]) / 1000
sm.links['true_time'] = sm.links['time'].copy()
sm.links['time'] = sm.links['time'] + (sm.links['km_fare'] / VALUE_OF_TIME_SEC)

In [16]:
# pruning algorithm
sm.step_pt_pathfinder(
    broken_modes=True, broken_routes=True,
    walk_on_road=True, keep_pathfinder=True
)

# remove intrazonal (handled differently)
sm.pt_los = sm.pt_los[sm.pt_los['origin']!=sm.pt_los['destination']]
sm.pt_los.reset_index(drop=True, inplace=True)

start publicpathfinder
68 sources 68 targets direct search
{'route_type': [{'rail'}, {'bus'}, {'rail', 'bus'}, {'tramway'}, {'rail', 'tramway'}, {'bus', 'tramway'}, {'rail', 'bus', 'tramway'}]}
3 + 9 = 12 < 0.5 * 68 splitting od_set
7 + 68 = 75 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
0 + 2 = 2 >= 0.5 * 2 keeping od_set
3 + 2 = 5 < 0.5 * 28 splitting od_set
1 + 1 = 2 < 0.5 * 64 splitting od_set
3 + 1 = 4 >= 0.5 * 5 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
3 + 4 = 7 >= 0.5 * 7 keeping od_set
7 + 6 = 13 < 0.5 * 42 splitting od_set
10 + 22 = 32 < 0.5 * 68 splitting od_set
1 + 1 = 2 < 0.5 * 68 splitting od_set
2 + 1 = 3 >= 0.5 * 4 keeping od_set
6 + 5 = 11 >= 0.5 * 18 keeping od_set
5 + 5 = 10 < 0.5 * 68 splitting od_set
1 + 2 = 3 < 0.5 * 9 splitting od_set
1 + 1 = 2

  0%|          | 0/100 [00:00<?, ?it/s]

49 sources 2 targets transposed search


route_id {'16_TBM_3'}:   2%|▏         | 2/100 [00:01<00:52,  1.86it/s]                         

3 sources 41 targets direct search
25 sources 2 targets transposed search


route_id {'25_TBM_3'}:   3%|▎         | 3/100 [00:01<00:41,  2.32it/s]

... (162 outputs left out)

68 sources 68 targets direct search


route_type {'rail', 'bus'}: 100%|██████████| 7/7 [00:17<00:00,  2.55s/it]


path_analysis


path_analysis: 100%|██████████| 32291/32291 [00:09<00:00, 3491.21it/s]


In [17]:
# pruning algorithm
sm.step_pt_pathfinder(
    broken_modes=False, broken_routes=False,
    walk_on_road=True, keep_pathfinder=True
)

# remove intrazonal (handled differently)
sm.pt_los = sm.pt_los[sm.pt_los['origin']!=sm.pt_los['destination']]
sm.pt_los.reset_index(drop=True, inplace=True)

start publicpathfinder
68 sources 68 targets direct search
path_analysis


path_analysis: 100%|██████████| 4624/4624 [00:00<00:00, 8718.40it/s]


In [18]:
hierarchy = ['rail', 'tramway', 'bus', 'schoolbus', 'ferry', 'walk']
sm.analysis_pt_route_type(hierarchy)

In [19]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm.links.route_type.unique():
    links = sm.links[sm.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm.pt_los[k + '_link_path'] = sm.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm.pt_los[k + '_in_vehicle_length'] = sm.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm.pt_los['fare'] = 0
sm.pt_los['flat_fare'] = 0
sm.pt_los['km_fare'] = 0

for m in links_mode.keys():
    sm.pt_los['fare'] += (sm.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000
    sm.pt_los['flat_fare'] += (sm.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m]
    sm.pt_los['km_fare'] += sm.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

sm.pt_los['parking_fare'] = 0

# PATHFINDER - PT + BIKE

In [20]:
sm_bike_pt = sm.copy()

In [21]:
### Connexion possible seulement avec le train, créer des liens d'accès à vitesse vélo et 
# des liens de diffusion à vitesse de marche (les gens laissent leur vélo à la gare de départ)
# Pour le moment on ne calcule pas l'utilité des connecteurs vélo en coût généralisé mais simplement en temps.

In [22]:
### Idée développer un réseau d'accès et un réseau de diffusion.
# Tous les connecteurs en accès rejoignent le réseau d'accès (vélo)
# Tous les connecteurs en diffusion partent du réseau de diffusion (marche à pied)
# Tous les connecteurs au réseau routier existent pour les deux réseaux
# --> problème : plus personne ne prend le train, l'attente est trop importante, il est plus interessant de se déplacer en vélo
# --             jusqu'à la destination finale.

### sm_bike_pt.road_links = sm_bike_pt.road_links.loc[sm_bike_pt.road_links['bike']]
### sm_bike_pt.road_nodes = sm_bike_pt.road_nodes.loc[sm_bike_pt.road_nodes['bike']]
### 
### sm_bike_pt.links = sm_bike_pt.links.loc[sm_bike_pt.links['route_type'] == 'rail']
### sm_bike_pt.nodes = sm_bike_pt.nodes.loc[list(set(sm_bike_pt.links['a'].values.tolist() + sm_bike_pt.links['b'].values.tolist()))]
### 
### sm_bike_pt.integrity_fix_road_network()
### 
### sm_bike_pt.preparation_ntlegs(
###     zone_to_road=True,
###     zone_to_transit=False,
###     road_to_transit=True
### )
### 
### sm_bike_pt._add_type_prefixes(
###     prefixes={
###         'zone_to_road': 'ztr_',
###         'road_to_transit': 'rtt_',
###     }
### )
### 
### inbound_road_links = sm_bike_pt.road_links.copy()
### inbound_road_nodes = sm_bike_pt.road_nodes.copy()
### road_to_transit_inbound = sm_bike_pt.road_to_transit.copy()
### 
### ### Rlinks
### inbound_road_links.index += '_a'
### inbound_road_links['a'] += '_a'
### inbound_road_links['b'] += '_a'
### ### TODO : ajouter la définition du temps en fonction des coefficients kappa
### inbound_road_links['walk_speed'] = inbound_road_links.apply(lambda x: max(x['speed']*BIKE_SPEED['bike_speed_factor_no_infra'], BIKE_SPEED['max_bike_speed_no_infra']) if (x['bike_index'] < 2) else BIKE_SPEED['bike_speed_infra'], 1)
### inbound_road_links['walk_time'] = inbound_road_links['length']/inbound_road_links['speed']*3.6
### 
### ### Rnodes
### inbound_road_nodes.index += '_a'
### 
### ### RoadToTransit
### road_to_transit_inbound.index += '_a'
### road_to_transit_inbound.loc[road_to_transit_inbound['direction'] == 'access', 'b'] += '_a'
### road_to_transit_inbound.loc[road_to_transit_inbound['direction'] == 'eggress', 'a'] += '_a'
### 
### ### ZoneToTransit
### sm_bike_pt.zone_to_road.loc[sm_bike_pt.zone_to_road['direction'] == 'access', 'b'] += '_a'
### 
### ### Concat
### sm_bike_pt.road_links = pd.concat([sm_bike_pt.road_links, inbound_road_links])
### sm_bike_pt.road_nodes = pd.concat([sm_bike_pt.road_nodes, inbound_road_nodes])
### sm_bike_pt.road_to_transit = pd.concat([sm_bike_pt.road_to_transit, road_to_transit_inbound])
### 
### ### PathFinder
### sm_bike_pt.step_pt_pathfinder(
###     broken_modes=False, broken_routes=False,
###     walk_on_road=True
### )
### sm_bike_pt.analysis_pt_route_type(['rail', 'bike'])
### 
### # remove intrazonal (handled differently)
### sm_bike_pt.pt_los = sm_bike_pt.pt_los[sm_bike_pt.pt_los['origin']!=sm_bike_pt.pt_los['destination']]
### sm_bike_pt.pt_los.reset_index(drop=True, inplace=True)

In [23]:
### Idée utiliser les deux tailles de connecteurs

### sm_bike_pt.preparation_ntlegs(
###     zone_to_road=False,
###     zone_to_transit=True,
###     road_to_transit=False,
###     n_ntlegs = 10,
###     max_ntleg_length = 10000
### )
### 
### sm_bike_pt._add_type_prefixes(
###     prefixes={
###         'zone_to_transit': 'ztt_'
###     }
### )
### 
### sm_bike_pt.preparation_footpaths(speed=WALK_SPEED/WALK_TIME_FACTOR, max_length=500)
### 
### sm_bike_pt.zone_to_transit.loc[sm_bike_pt.zone_to_transit['direction'] == 'access', 'speed'] = BIKE_SPEED['max_bike_speed_no_infra']/BIKE_TIME_FACTOR
### sm_bike_pt.zone_to_transit.loc[sm_bike_pt.zone_to_transit['direction'] == 'eggress', 'speed'] = WALK_SPEED/WALK_TIME_FACTOR
### 
### sm_bike_pt.step_pt_pathfinder(
###     broken_modes=False, broken_routes=False,
###     walk_on_road=True
### )
### 
### sm_bike_pt.analysis_pt_route_type(['rail', 'bike'])
### 
### # remove intrazonal (handled differently)
### sm_bike_pt.pt_los = sm_bike_pt.pt_los[sm_bike_pt.pt_los['origin']!=sm_bike_pt.pt_los['destination']]
### sm_bike_pt.pt_los.reset_index(drop=True, inplace=True)

In [24]:
sm_bike_pt.pt_los = sm_bike_pt.pt_los.loc[sm_bike_pt.pt_los['route_type'] == 'rail']

In [25]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm_bike_pt.links.route_type.unique():
    links = sm_bike_pt.links[sm_bike_pt.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm_bike_pt.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm_bike_pt.pt_los[k + '_link_path'] = sm_bike_pt.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm_bike_pt.pt_los[k + '_in_vehicle_length'] = sm_bike_pt.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm_bike_pt.pt_los['fare'] = 0
sm_bike_pt.pt_los['km_fare'] = 0
sm_bike_pt.pt_los['flat_fare'] = 0

for m in links_mode.keys():
    sm_bike_pt.pt_los['fare'] += (sm_bike_pt.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm_bike_pt.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000
    sm_bike_pt.pt_los['flat_fare'] += (sm_bike_pt.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m]
    sm_bike_pt.pt_los['km_fare'] += sm_bike_pt.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

sm_bike_pt.pt_los['parking_fare'] = 0

# PATHFINDER PT + VP

In [26]:
sm_car_pt = sm.copy()

In [27]:
### Connexion possible seulement avec le train, créer des liens d'accès à vitesse vélo et 
# des liens de diffusion à vitesse de marche (les gens laissent leur vélo à la gare de départ)
# Pour le moment on ne calcule pas l'utilité des connecteurs vélo en coût généralisé mais simplement en temps.

In [28]:
### Idée développer un réseau d'accès et un réseau de diffusion.
# Tous les connecteurs en accès rejoignent le réseau d'accès (vélo)
# Tous les connecteurs en diffusion partent du réseau de diffusion (marche à pied)
# Tous les connecteurs au réseau routier existent pour les deux réseaux

In [29]:
sm_car_pt.links = sm_car_pt.links.loc[sm_car_pt.links['route_type'] == 'rail']
sm_car_pt.nodes = sm_car_pt.nodes.loc[list(set(sm_car_pt.links['a'].values.tolist() + sm_car_pt.links['b'].values.tolist()))]

sm_car_pt.integrity_fix_road_network()

direct: 58184it [00:07, 7388.83it/s]
reversed: 58184it [00:07, 7683.65it/s] 
58184it [00:07, 7737.26it/s] 


In [30]:
sm_car_pt.preparation_ntlegs(
    zone_to_road=True,
    zone_to_transit=False,
    road_to_transit=True
)

sm_car_pt._add_type_prefixes(
    prefixes={
        'zone_to_road': 'ztr_',
        'road_to_transit': 'rtt_',
    }
)

In [31]:
inbound_road_links = sm_car_pt.road_links.copy()
inbound_road_nodes = sm_car_pt.road_nodes.copy()
road_to_transit_inbound = sm_car_pt.road_to_transit.copy()

In [32]:
inbound_road_links = inbound_road_links.loc[inbound_road_links['car']]
inbound_road_links.index += '_a'
inbound_road_links['a'] += '_a'
inbound_road_links['b'] += '_a'

inbound_road_links['walk_speed'] = inbound_road_links['speed']
inbound_road_links['walk_time'] = inbound_road_links['time']

In [33]:
inbound_road_nodes = inbound_road_nodes.loc[inbound_road_nodes['car']]
inbound_road_nodes.index += '_a'

In [34]:
road_to_transit_inbound.index += '_a'
road_to_transit_inbound.loc[road_to_transit_inbound['direction'] == 'access', 'b'] += '_a'
road_to_transit_inbound.loc[road_to_transit_inbound['direction'] == 'eggress', 'a'] += '_a'

In [35]:
sm_car_pt.zone_to_road.loc[sm_car_pt.zone_to_road['direction'] == 'access', 'b'] += '_a'

In [36]:
sm_car_pt.road_links = pd.concat([sm_car_pt.road_links, inbound_road_links])
sm_car_pt.road_nodes = pd.concat([sm_car_pt.road_nodes, inbound_road_nodes])
sm_car_pt.road_to_transit = pd.concat([sm_car_pt.road_to_transit, road_to_transit_inbound])

In [37]:
sm_car_pt.step_pt_pathfinder(
    broken_modes=False, broken_routes=False,
    walk_on_road=True
)

sm_car_pt.analysis_pt_route_type(['car', 'bike'])

# remove intrazonal (handled differently)
sm_car_pt.pt_los = sm_car_pt.pt_los[sm_car_pt.pt_los['origin']!=sm_car_pt.pt_los['destination']]
sm_car_pt.pt_los.reset_index(drop=True, inplace=True)

start publicpathfinder
68 sources 68 targets direct search
path_analysis


path_analysis: 100%|██████████| 4289/4289 [00:01<00:00, 3735.96it/s]


In [38]:
sm_car_pt.pt_los = sm_car_pt.pt_los.loc[sm_car_pt.pt_los['route_type'] == 'rail']

In [39]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm_car_pt.links.route_type.unique():
    links = sm_car_pt.links[sm_car_pt.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm_car_pt.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm_car_pt.pt_los[k + '_link_path'] = sm_car_pt.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm_car_pt.pt_los[k + '_in_vehicle_length'] = sm_car_pt.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm_car_pt.pt_los['fare'] = 0
sm_car_pt.pt_los['km_fare'] = 0
sm_car_pt.pt_los['flat_fare'] = 0

for m in links_mode.keys():
    sm_car_pt.pt_los['fare'] += (sm_car_pt.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm_car_pt.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000
    sm_car_pt.pt_los['flat_fare'] += (sm_car_pt.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m]
    sm_car_pt.pt_los['km_fare'] += sm_car_pt.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

sm_car_pt.pt_los['parking_fare'] = 0 ### Parking gratuit au parking relais ?

# Correct times

In [40]:
# TC
sm.links['pathfinder_time'] = sm.links['time']
sm.links['time'] = sm.links['true_time']
sm.road_links['walk_time'] *= 1 / WALK_TIME_FACTOR
sm.zone_to_road['time'] *= 1 / WALK_TIME_FACTOR
sm.analysis_pt_time(boarding_time=0, walk_on_road=True)

# Vélo + TC
### TODO: séparer le temps marché (diffusion) du temps à vélo (accès)
sm_bike_pt.links['pathfinder_time'] = sm_bike_pt.links['time']
sm_bike_pt.links['time'] = sm_bike_pt.links['true_time']
sm_bike_pt.zone_to_road['time'] *= 1 / BIKE_TIME_FACTOR
sm_bike_pt.analysis_pt_time(boarding_time=0, walk_on_road=True)

### TODO: CAR + TC
sm_car_pt.links['pathfinder_time'] = sm_car_pt.links['time']
sm_car_pt.links['time'] = sm_car_pt.links['true_time']
sm_car_pt.analysis_pt_time(boarding_time=0, walk_on_road=True)

In [41]:
sm.pt_los = pd.concat([sm.pt_los, sm_bike_pt.pt_los, sm_car_pt.pt_los])

# ROAD PATHFINDER | car

In [42]:
# empty volumes
sm.volumes = pd.DataFrame(index=pd.MultiIndex.from_product([sm.zones.index]*2))
sm.volumes.index.names = ['origin', 'destination']
sm.volumes.reset_index(inplace=True)

In [43]:
sm.step_road_pathfinder(method='aon')
# remove intrazonal (handled differently)
sm.car_los = sm.car_los[sm.car_los['origin']!=sm.car_los['destination']]
sm.car_los['pathfinder_session'] = 'car_aon'

In [44]:
sm.analysis_car_los()
sm.analysis_car_length()

# ROAD PATHFINDER | bike

In [45]:
sm_bike = sm.copy()

In [46]:
# Pour l'instant, on ne fait le pathfinder qu'avec un plus court chemin facile, ensuite on intégrera 
# les constantes de Royston en vitesse ce qui permettra de faire quelque chose de super.

In [47]:
sm_bike.road_links['speed'] = sm_bike.road_links.apply(lambda x: max(x['speed']*BIKE_SPEED['bike_speed_factor_no_infra'], BIKE_SPEED['max_bike_speed_no_infra']) if (x['bike_index'] < 2) else BIKE_SPEED['bike_speed_infra'], 1)
sm_bike.road_links['time'] = sm_bike.road_links['length']/sm_bike.road_links['speed']*3.6

In [48]:
# empty volumes
sm_bike.volumes = pd.DataFrame(index=pd.MultiIndex.from_product([sm.zones.index]*2))
sm_bike.volumes.index.names = ['origin', 'destination']
sm_bike.volumes.reset_index(inplace=True)

In [49]:
sm_bike.step_road_pathfinder(method='aon')
# remove intrazonal (handled differently)
sm_bike.car_los = sm_bike.car_los[sm_bike.car_los['origin']!=sm_bike.car_los['destination']]
sm_bike.car_los['pathfinder_session'] = 'bike_aon'

In [50]:
sm_bike.analysis_car_los()
sm_bike.analysis_car_length()

# INTRAZONAL

In [51]:
sm.zone_length = (sm.zones.area ** 0.5) * INTRA_AREA_TO_DISTANCE_FACTOR

In [52]:
def set_intra_los(mode, speed_kmh, km_fare, base_fare=0, time_penalty=0):
    intra_los = sm.volumes[sm.volumes['origin']==sm.volumes['destination']].copy()

    intra_los['route_type'] = mode
    if mode != 'walk':
        intra_los['route_types'] = [(mode,)]*len(intra_los)
    else:
        intra_los['route_types'] = [tuple()]*len(intra_los)

    intra_los['length'] = intra_los['origin'].apply(lambda x: sm.zone_length[x])
    intra_los['gtime'] = intra_los['origin'].apply(
        lambda x: sm.zone_length[x] / (speed_kmh / 3.6)
    )
    intra_los['gtime'] += time_penalty
    intra_los['time'] = intra_los['gtime']
    intra_los['fare'] = intra_los['origin'].apply(
        lambda x: sm.zone_length[x] * km_fare / 1000 + base_fare
    )
    intra_los['pathfinder_session'] = 'intrazonal'
    intra_los['ntransfers'] = 0
    intra_los['path'] = intra_los['origin'].apply(lambda x: tuple([x]))

    for col in ['node_path', 'link_path', 'boardings', 'alightings',
        'footpaths', 'ntlegs', 'boarding_links', 'alighting_links', 'transfers']:
            intra_los[col] = [tuple()]*len(intra_los)

    return intra_los

sm.intra_los = pd.concat(
    [
        set_intra_los('car', 40/CAR_TIME_FACTOR, KM_FARES['car'], base_fare=0),
        set_intra_los('bike', BIKE_SPEED['max_bike_speed_no_infra']/BIKE_TIME_FACTOR, 0),
        set_intra_los('walk', WALK_SPEED/WALK_TIME_FACTOR, 0)
    ]
)

sm.intra_los.reset_index(drop=True, inplace=True)

In [53]:
sm.intra_los['flat_fare'] = 0
sm.intra_los['km_fare'] = sm.intra_los['fare'].copy()

sm.intra_los['parking_fare'] = 0
sm.intra_los.loc[sm.intra_los['route_type'] == 'car', 'parking_fare'] = sm.intra_los.loc[sm.intra_los['route_type'] == 'car', 'destination'].map(sm.zones['parking_fare'].to_dict())
sm.intra_los['fare'] += sm.intra_los['parking_fare']

# CONCAT

In [54]:
# CAR
sm.car_los['penalties'] = 0
sm.car_los['ntransfers'] = 0
sm.car_los['parking_fare'] = sm.car_los['destination'].map(sm.zones['parking_fare'].to_dict())
sm.car_los['flat_fare'] = 0
sm.car_los['km_fare'] = sm.car_los['length'] * KM_FARES['car'] / 1000 
sm.car_los['fare'] = sm.car_los['length'] * KM_FARES['car'] / 1000 + sm.car_los['parking_fare']
sm.car_los['gtime'] *= CAR_TIME_FACTOR
sm.car_los['gtime'] += sm.car_los['penalties']
sm.car_los['route_types'] = [('car',)] * len(sm.car_los)
sm.car_los['route_type'] = 'car'
sm.car_los['time'] = sm.car_los['gtime']
sm.car_los['gtime'] += sm.car_los['in_vehicle_length'] / 1000 * KM_FARES['car'] / VALUE_OF_TIME_SEC

# BIKE
sm_bike.car_los['penalties'] = 0
sm_bike.car_los['ntransfers'] = 0
sm_bike.car_los['parking_fare'] = 0
sm_bike.car_los['flat_fare'] = 0
sm_bike.car_los['km_fare'] = 0
sm_bike.car_los['fare'] = 0
sm_bike.car_los['gtime'] *= BIKE_TIME_FACTOR
sm_bike.car_los['gtime'] += sm_bike.car_los['penalties']
sm_bike.car_los['route_types'] = [('bike',)] * len(sm.car_los)
sm_bike.car_los['route_type'] = 'bike'
sm_bike.car_los['time'] = sm.car_los['gtime']

sm.los = pd.concat([sm.car_los, sm_bike.car_los, sm.pt_los, sm.intra_los]).reset_index(drop=True)

# EXPORT

In [55]:
print(sm.los['route_types'].unique())

sm.to_zippedpickles(
    mpath + r'pathfinders',
    omitted_attributes=['pt_los', 'car_los', 'intra_los']
)

[('car',) ('bike',) () ('bus', 'tramway') ('bus',) ('bus', 'rail')
 ('bus', 'rail', 'tramway') ('rail',) ('tramway',)]


  0%|          | 0/25 [00:00<?, ?it/s]

los: 100%|██████████| 25/25 [00:12<00:00,  2.08it/s]             


In [56]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined